In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [3]:
df_customers = spark.read.csv('../data/raw/olist_customers_dataset.csv',header=True,escape="\"")
df_customers.printSchema()
df_customers.count()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



99441

In [4]:
df_geolocation = spark.read.csv('../data/raw/olist_geolocation_dataset.csv',header=True,escape="\"")
df_geolocation.printSchema()
df_geolocation.count()

root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: string (nullable = true)
 |-- geolocation_lng: string (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)



1000163

In [5]:
df_order_items = spark.read.csv('../data/raw/olist_order_items_dataset.csv',header=True,escape="\"")
df_order_items.printSchema()
df_order_items.count()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- freight_value: string (nullable = true)



112650

In [6]:
df_order_payments = spark.read.csv('../data/raw/olist_order_payments_dataset.csv',header=True,escape="\"")
df_order_payments.printSchema()
df_order_payments.count()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: string (nullable = true)
 |-- payment_value: string (nullable = true)



103886

In [7]:
df_order_reviews = spark.read.csv(
    '../data/raw/olist_order_reviews_dataset.csv',
    header=True,
    escape='"',
    multiLine=True
)
df_order_reviews.printSchema()
df_order_reviews.count()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)



99224

In [8]:
df_orders = spark.read.csv('../data/raw/olist_orders_dataset.csv',header=True,escape="\"")
df_orders.printSchema()
df_orders.count()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)



99441

In [9]:
df_products = spark.read.csv('../data/raw/olist_products_dataset.csv',header=True,escape="\"")
df_products.printSchema()
df_products.count()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: string (nullable = true)
 |-- product_description_lenght: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)



32951

In [10]:
df_sellers = spark.read.csv('../data/raw/olist_sellers_dataset.csv',header=True,escape="\"")
df_sellers.printSchema()
df_sellers.count()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



3095

In [11]:
df_category = spark.read.csv('../data/raw/product_category_name_translation.csv',header=True,escape="\"")
df_category.printSchema()
df_category.count()

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



71

In [12]:
df_orders.write.parquet('../data/bronze/orders', mode='overwrite')
df_geolocation.write.parquet('../data/bronze/geolocation', mode='overwrite')
df_category.write.parquet('../data/bronze/category', mode='overwrite')
df_customers.write.parquet('../data/bronze/customers', mode='overwrite')
df_order_items.write.parquet('../data/bronze/order_items', mode='overwrite')
df_order_payments.write.parquet('../data/bronze/order_payments', mode='overwrite')
df_order_reviews.write.parquet('../data/bronze/order_reviews', mode='overwrite')
df_products.write.parquet('../data/bronze/products', mode='overwrite')
df_sellers.write.parquet('../data/bronze/sellers', mode='overwrite')

26/06/17 11:58:54 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


In [13]:
from collections import defaultdict

dataframes = {
    'orders': df_orders,
    'order_items': df_order_items,
    'order_payments': df_order_payments,
    'order_reviews': df_order_reviews,
    'customers': df_customers,
    'products': df_products,
    'sellers': df_sellers,
    'category': df_category,
    'geolocation': df_geolocation
}

col_to_tables = defaultdict(list)

for name, df in dataframes.items():
    for col in df.columns:
        col_to_tables[col].append(name)

for col, tables in col_to_tables.items():
    if len(tables) > 1:
        print(f"{col} --- {', '.join(tables)}")

order_id --- orders, order_items, order_payments, order_reviews
customer_id --- orders, customers
product_id --- order_items, products
seller_id --- order_items, sellers
product_category_name --- products, category
